# HY-World 2.0 — Реконструкция 3D-мира аэропорта

**Модель:** [tencent/HY-World-2.0](https://huggingface.co/tencent/HY-World-2.0)  
**Что делает:** принимает несколько кадров видео → строит 3D сцену в виде Gaussian Splatting / point cloud.

### Два режима:
| Режим | Вход | Выход |
|---|---|---|
| **World Reconstruction** | N кадров видео / мультивью изображений | 3DGS, point cloud, depth |
| **World Generation** | текст или одно изображение | полная 3D сцена |

## 1. Проверка окружения

In [ ]:
import subprocess, sys, platform

print(f"Python : {sys.version}")
print(f"OS     : {platform.system()} {platform.release()}")

# CUDA check
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA   : {torch.version.cuda}")
    if torch.cuda.is_available():
        dev = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU    : {dev}  ({vram:.1f} GB VRAM)")
        if vram < 16:
            print("⚠️  HY-World 2.0 рекомендует >= 16 GB VRAM (WorldMirror ~1.2B params)")
            print("   На меньшем GPU можно попробовать режим fp16 или обрезать входные кадры")
    else:
        print("❌ CUDA недоступна — HY-World требует GPU")
except ImportError:
    print("❌ PyTorch не установлен")

# nvidia-smi
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                        '--format=csv,noheader'], capture_output=True, text=True)
    print("\nnvidia-smi:", r.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi не найден")

## 2. Установка HY-World 2.0

> Клонируем репозиторий и устанавливаем зависимости.  
> **Один раз** — повторно запускать не нужно.

In [ ]:
import os

REPO_DIR = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0"

if not os.path.isdir(REPO_DIR):
    print("Клонируем репозиторий...")
    os.system(f'git clone https://github.com/Tencent-Hunyuan/HY-World-2.0 "{REPO_DIR}"')
else:
    print("Репозиторий уже скачан:", REPO_DIR)

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

In [ ]:
# Устанавливаем зависимости
# flash-attn может долго компилироваться (5-15 мин), можно пропустить если нет компилятора

!pip install -q -r requirements.txt

# flash-attn — опционально (нужен C++ компилятор)
try:
    import flash_attn
    print("✅ flash-attn уже установлен:", flash_attn.__version__)
except ImportError:
    print("flash-attn не установлен. Пробуем установить pre-built wheel...")
    import subprocess, sys, torch
    cuda_ver = torch.version.cuda.replace('.', '')
    py_ver   = f"{sys.version_info.major}{sys.version_info.minor}"
    # Пробуем pre-built wheel с GitHub releases
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'flash-attn', '--no-build-isolation', '-q'],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print("✅ flash-attn установлен")
    else:
        print("⚠️  flash-attn не удалось установить — будет использован стандартный attention")
        print(r.stderr[-500:])

## 3. Подготовка входных кадров аэропорта

Извлекаем кадры из аэропортового видео.  
WorldMirror работает лучше всего с **10-30 кадрами** с плавным движением камеры.

In [ ]:
import cv2
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Путь к видео (замени на своё)
VIDEO_PATH = r"C:\Users\shche\Desktop\Application_for_models\137826-video.h264 (online-video-cutter.com).mp4"
OUTPUT_DIR = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\inputs\airport"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_FRAMES   = 24    # сколько кадров извлечь
MAX_SIZE   = 1024  # максимальная сторона (HY-World принимает 50K-500K пикселей)

cap = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps   = cap.get(cv2.CAP_PROP_FPS)
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Видео: {total} кадров, {fps:.1f} fps, {w}×{h}")

# Равномерно распределённые индексы
indices = np.linspace(0, total - 1, N_FRAMES, dtype=int)
saved   = []

for i, idx in enumerate(indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, frame = cap.read()
    if not ret:
        continue
    # Масштабируем если нужно
    if max(frame.shape[:2]) > MAX_SIZE:
        scale  = MAX_SIZE / max(frame.shape[:2])
        frame  = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    path = os.path.join(OUTPUT_DIR, f"frame_{i:04d}.jpg")
    cv2.imwrite(path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    saved.append(path)

cap.release()
print(f"Сохранено {len(saved)} кадров → {OUTPUT_DIR}")

# Визуализируем сетку
cols = 6
rows = (len(saved) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*2.5))
axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
for ax, p in zip(axes.flatten() if hasattr(axes, 'flatten') else [axes], saved):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(Path(p).name, fontsize=7)
for ax in axes.flatten()[len(saved):]:
    ax.axis('off')
plt.suptitle(f"Входные кадры для HY-World 2.0 ({len(saved)} шт.)", fontsize=12)
plt.tight_layout()
plt.savefig("input_frames_preview.png", dpi=100, bbox_inches='tight')
plt.show()

## 4. World Reconstruction — реконструкция 3D сцены из кадров

WorldMirror 2.0 принимает набор изображений и строит:
- **Depth maps** — карты глубины для каждого кадра
- **Point cloud** — облако точек сцены
- **3DGS** (3D Gaussian Splatting) — фотореалистичное 3D представление

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0")

from hyworld2.worldrecon.pipeline import WorldMirrorPipeline
import torch

print("Загружаем WorldMirror 2.0...")
pipeline = WorldMirrorPipeline.from_pretrained(
    'tencent/HY-World-2.0',
    torch_dtype=torch.bfloat16,   # экономия VRAM
)
pipeline = pipeline.to('cuda')
print("✅ Модель загружена")

# Показываем сколько занимает VRAM
allocated = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM занято: {allocated:.2f} GB")

In [ ]:
import time

OUTPUT_3D = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\outputs\airport_recon"
os.makedirs(OUTPUT_3D, exist_ok=True)

print(f"Запускаем реконструкцию {len(saved)} кадров...")
t0 = time.time()

result = pipeline(
    OUTPUT_DIR,           # папка с входными изображениями
    output_dir=OUTPUT_3D,
)

print(f"\n✅ Реконструкция завершена за {time.time()-t0:.0f} сек")
print("Выходные файлы:")
for f in Path(OUTPUT_3D).rglob('*'):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {f.name:40s} {size:.0f} KB")

## 5. Визуализация Point Cloud

In [ ]:
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

# Ищем файл point cloud в выходах
cloud_files = list(Path(OUTPUT_3D).rglob('*.ply')) + \
              list(Path(OUTPUT_3D).rglob('*.pcd')) + \
              list(Path(OUTPUT_3D).rglob('*point_cloud*'))

print("Найденные файлы:", cloud_files)

if cloud_files:
    # Попробуем загрузить .ply через open3d или trimesh
    try:
        import open3d as o3d
        pcd = o3d.io.read_point_cloud(str(cloud_files[0]))
        pts = np.asarray(pcd.points)
        colors = np.asarray(pcd.colors) if pcd.has_colors() else None
        print(f"Point cloud: {len(pts)} точек")
    except ImportError:
        # fallback: PLY через trimesh
        try:
            import trimesh
            mesh = trimesh.load(str(cloud_files[0]))
            pts    = mesh.vertices if hasattr(mesh, 'vertices') else np.array(mesh.points)
            colors = None
            print(f"Загружено {len(pts)} точек (trimesh)")
        except:
            print("Установите open3d: pip install open3d")
            pts = None

    if pts is not None:
        # Subsample для Plotly (max 50k точек)
        step = max(1, len(pts) // 50000)
        pts_s = pts[::step]
        col_s = (colors[::step] * 255).astype(int) if colors is not None else None

        marker_kwargs = dict(size=1.5, opacity=0.7)
        if col_s is not None:
            marker_kwargs['color'] = [f'rgb({r},{g},{b})' for r,g,b in col_s]
        else:
            marker_kwargs['color'] = pts_s[:, 2]   # Z как цвет
            marker_kwargs['colorscale'] = 'Viridis'

        fig = go.Figure(go.Scatter3d(
            x=pts_s[:,0], y=pts_s[:,1], z=pts_s[:,2],
            mode='markers', marker=marker_kwargs,
            name=f'Airport point cloud ({len(pts_s):,} pts)'
        ))
        fig.update_layout(
            title='HY-World 2.0 — Airport Point Cloud',
            scene=dict(
                bgcolor='#0f172a',
                xaxis=dict(gridcolor='#334155'),
                yaxis=dict(gridcolor='#334155'),
                zaxis=dict(gridcolor='#334155'),
            ),
            paper_bgcolor='#0f172a',
            font=dict(color='white'),
            height=600,
        )
        fig.write_html('airport_point_cloud.html', include_plotlyjs='cdn')
        fig.show()
        print("Сохранено: airport_point_cloud.html")
else:
    print("Файлы point cloud не найдены. Проверьте формат выходных данных модели.")

## 6. Визуализация Depth Maps

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2

# Ищем depth maps (.npy, .png, .exr)
depth_files = sorted(
    list(Path(OUTPUT_3D).rglob('*depth*')) +
    list(Path(OUTPUT_3D).rglob('*disp*'))
)[:12]  # первые 12

if depth_files:
    cols = 4
    rows = (len(depth_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, f in zip(axes_flat, depth_files):
        f = Path(f)
        if f.suffix == '.npy':
            depth = np.load(str(f))
        else:
            depth = cv2.imread(str(f), cv2.IMREAD_ANYDEPTH)
        if depth is None:
            ax.axis('off'); continue
        # Нормализуем
        d_norm = depth.astype(float)
        d_norm = (d_norm - d_norm.min()) / (d_norm.max() - d_norm.min() + 1e-9)
        ax.imshow(d_norm, cmap='inferno')
        ax.set_title(f.name, fontsize=7)
        ax.axis('off')

    for ax in axes_flat[len(depth_files):]:
        ax.axis('off')

    plt.suptitle('Depth Maps — Airport Scene', fontsize=13)
    plt.tight_layout()
    plt.savefig('airport_depth_maps.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Сохранено: airport_depth_maps.png")
else:
    print("Depth maps не найдены. Проверьте выходные файлы модели.")

## 7. World Generation — генерация сцены из текста

Второй режим: вместо видео подаём текстовый промпт → модель генерирует 3D мир аэропорта с нуля.

In [ ]:
# World Generation pipeline (текст → 3D)
try:
    from hyworld2.worldgen.pipeline import WorldGenPipeline

    gen_pipeline = WorldGenPipeline.from_pretrained(
        'tencent/HY-World-2.0',
        torch_dtype=torch.bfloat16,
    ).to('cuda')

    PROMPT = (
        "Airport apron with commercial aircraft, ground crew, "
        "jet bridge, taxiway markings, sunny day, high detail, photorealistic"
    )

    OUTPUT_GEN = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\outputs\airport_gen"
    os.makedirs(OUTPUT_GEN, exist_ok=True)

    print(f"Генерируем 3D мир по промпту:\n  '{PROMPT}'")
    t0 = time.time()

    result_gen = gen_pipeline(
        prompt=PROMPT,
        output_dir=OUTPUT_GEN,
        num_inference_steps=50,
    )

    print(f"\n✅ Генерация завершена за {time.time()-t0:.0f} сек")
    print("Выходные файлы:")
    for f in Path(OUTPUT_GEN).rglob('*'):
        if f.is_file():
            print(f"  {f.name}")

except Exception as e:
    print(f"World Generation недоступен: {e}")
    print("Возможно этот режим требует отдельного компонента — проверь документацию.")

## 8. Просмотр 3DGS (Gaussian Splatting)

Если модель сохранила `.ply` файл с Gaussian Splatting, его можно открыть в:
- **SuperSplat** (веб, бесплатно): https://supersplat.playcanvas.com
- **KIRI Engine** (десктоп)
- **Luma AI Viewer**

Ниже — поиск и отображение информации о 3DGS файлах.

In [ ]:
from pathlib import Path
import os

# Ищем 3DGS / PLY файлы
search_dirs = [
    r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\outputs"
]

ply_files  = []
splat_files = []

for d in search_dirs:
    for f in Path(d).rglob('*'):
        if f.suffix.lower() == '.ply':
            ply_files.append(f)
        if f.suffix.lower() in ('.splat', '.3dgs'):
            splat_files.append(f)

print("PLY файлы:")
for f in ply_files:
    print(f"  {f}  ({f.stat().st_size / 1024**2:.1f} MB)")

print("\nGaussian Splat файлы:")
for f in splat_files:
    print(f"  {f}  ({f.stat().st_size / 1024**2:.1f} MB)")

if ply_files:
    best = max(ply_files, key=lambda f: f.stat().st_size)
    print(f"\n👉 Открой в SuperSplat:")
    print(f"   1. Зайди на https://supersplat.playcanvas.com")
    print(f"   2. Перетащи файл: {best}")
    print(f"   3. Исследуй 3D сцену аэропорта в браузере!")

## 9. Командная строка (альтернатива Python API)

Если Python API ещё нестабилен, можно запускать через CLI:

In [ ]:
# Альтернативный запуск через CLI (inference скрипт из репозитория)
import subprocess, os

REPO = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0"
INPUT  = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\inputs\airport"
OUTPUT = r"C:\Users\shche\Desktop\Application_for_models\HY-World-2.0\outputs\airport_cli"

# Ищем inference скрипт в репозитории
scripts = list(Path(REPO).rglob('infer*.py')) + \
          list(Path(REPO).rglob('run*.py')) + \
          list(Path(REPO).rglob('demo*.py'))

print("Доступные скрипты инференса:")
for s in scripts:
    print(f"  {s}")

# Пример запуска (раскомментируй нужный):
# cmd = [
#     "python", str(scripts[0]),
#     "--input",  INPUT,
#     "--output", OUTPUT,
#     "--enable_bf16",
# ]
# print("\nКоманда:", ' '.join(cmd))
# result = subprocess.run(cmd, cwd=REPO, capture_output=False)
# print("Return code:", result.returncode)

## Итоги

| Шаг | Результат |
|---|---|
| Извлечение кадров | `inputs/airport/*.jpg` |
| World Reconstruction | `outputs/airport_recon/` — point cloud, depth, 3DGS |
| Point Cloud viz | `airport_point_cloud.html` — интерактивный Plotly |
| Depth Maps viz | `airport_depth_maps.png` |
| World Generation | `outputs/airport_gen/` — 3D мир из текста |
| 3DGS просмотр | SuperSplat / Luma AI (загрузи .ply файл) |

### Требования для полноценной работы:
- **GPU**: ≥ 16 GB VRAM (RTX 3090 / A100 / H100)
- **CUDA**: 12.4
- **RAM**: ≥ 32 GB
- **Место на диске**: ~20 GB для модели + выходы

### Если VRAM < 16 GB:
- Уменьши `N_FRAMES` до 6-8
- Уменьши `MAX_SIZE` до 512
- Используй `torch_dtype=torch.float16` вместо `bfloat16`
- Добавь `device_map='auto'` для CPU offloading